# Hunting MicroMonsters Bioinformatic Analysis Workflow

## Convert long *.fast5 names to short names

This document shows a method to easily convert long file names from ONT sequencing raw data to a short-consecutive-number set of files.
Each file name will be composed of the container folder name + the number of the file starting from 0.

#### Step-by-step breakdown:

1. find . -type f: Finds all files (not directories) starting from the current directory (.) and going into all subdirectories recursively.

2. while read -r filepath: Processes each file path found by the find command one at a time.

3. dir=$(dirname "$filepath"): Extracts the directory path from the full file path. For example:
    - If filepath is ./subfolder/myfile.txt, dir becomes ./subfolder
    - If filepath is ./myfile.txt, dir becomes .

4. file=$(basename "$filepath"): Extracts just the filename from the full path.
    - From ./subfolder/myfile.txt, file becomes myfile.txt
5. Folder name logic:
    - If the file is in the root directory (dir = "."), use the current working directory's name
    - Otherwise, use the subdirectory's name as the prefix
6. Counters calculation
    - Counts how many files already exist in that directory with the naming pattern foldername_*
    - This ensures proper numbering even if the script is run multiple times

7. Extension handling: Extracts the file extension so it is kept in the new name.

8. File renaming: Moves the file to its new name with the pattern foldername_counter.extension


### Changing the name of each file in one folder with the format "folder_name + number + extension"

Change the name while keeping the original file extensions. This process uses bash kernel.

In [ ]:
#locate the folder with the files
cd ~/path/to/folder
counter=0
folder_name=$(basename "$PWD")
for file in *; do
    if [ -f "$file" ]; then
        # Get file extension
        ext="${file##*.}"
        # Only add extension if file had one
        if [ "$ext" != "$file" ]; then
            mv "$file" "${folder_name}_${counter}.${ext}"
        else
            mv "$file" "${folder_name}_${counter}"
        fi
        ((counter++))
    fi
done

### Changing the name of each file in all subdirectories of a folder with the format "folder_name + number + extension"

A modification from the previous script that can change the name of several subdirectories contained in a main folder.

In [ ]:
# locate the folder with the files
cd ~/path/to/folder

find . -type f | while read -r filepath; do
    # Get the directory and filename
    dir=$(dirname "$filepath")
    file=$(basename "$filepath")
    
    # Get folder name (use parent folder name or current folder for root)
    if [ "$dir" = "." ]; then
        folder_name=$(basename "$PWD")
    else
        folder_name=$(basename "$dir")
    fi
    
    # Count existing renamed files in this directory to continue numbering
    counter=$(find "$dir" -maxdepth 1 -name "${folder_name}_*" | wc -l)
    
    # Get file extension
    ext="${file##*.}"
    
    # Rename the file
    if [ "$ext" != "$file" ]; then
        mv "$filepath" "${dir}/${folder_name}_${counter}.${ext}"
    else
        mv "$filepath" "${dir}/${folder_name}_${counter}"
    fi
done

## Converting the fast5 files to pod5

In this section it is described the process to convert the fast5 files from the sequencing to pod5 to optimized computing performance. The data will be processed downstream using Guppy.

The general workflow for this section is:
1. Environment creation
2. Convert single-read fast5 files into multi-read fast5 reads; using the "ont-fast5-api" module tool.
3. Convert the multiread-fast5 to pod5
4. Basecalling using guppy and the pod5 files

### Fast5-to-pod5 environment creation

Instructions to create the environments and modules needed for each task of the workflow (all commands run on bash kernel).

In [ ]:
# Create the environment
conda create -n fast5-to-pod5 -y

In [ ]:
# Activate environment
conda activate fast5-to-pod5

In [ ]:
# Install ont-fast5-api tool
pip install ont-fast5-api

In [ ]:
# Install pod5 tool
pip install pod5

### Activate the fast5-to-pod5 environment which contains the pod5 module

In [ ]:
# Activate the fast5-to-pod5 conda environment if is not active yet
conda activate fast5-to-pod5

Verify that the environment is active and that the module pod5 is running by looking for the "ont-fast5-api" module

In [ ]:
conda list ont-fast5-api

### Convert single-read fast5 files into multi-read fast5

To create the multi-read files, the tool will create a single file with all the fast5 files in the working folder (4000). The tool requires the input folder path and the output folder. "--recursive" command will look in subdirectories to convert.

In [ ]:
single_to_multi_fast5 \
--input_path ~/path/to/folder/input \
--save_path ~/path/to/folder/output \
--recursive

### Converting multiread-fast5 files into pod5

The newly converted multiread-fast5 will be converted to pod5 files with the pod5 tool. The script is run on bash and includes the command "--one-to-one" to keep the order of the files. The command "--threads" specify the use of all the cores available to speed up the process.

In [ ]:
pod5 convert fast5 \
~/path/to/folder/input/*.fast5 \
--output ~/path/to/folder/output \
--one-to-one ~/path/to/folder/input \
--threads 8

In [ ]:
# Deactivate the fast5-to-pod5 environment once the process is finished
conda deactivate

## Basecalling with Guppy from pod5 files

Now that we have the pod5 files on a folder, we will use Guppy tool for the basecalling. The script is run on bash using the "guppy" environment

Download the guppy file "ont_guppy_cpu_6.5.7-1~focal_amd64.deb" from the ONT community official webpage:
https://nanoporetech.com/software/other/guppy 

In [ ]:
# Create the guppy environment
conda create -n guppy

In [ ]:
# Activate the guppy environment
conda activate guppy

In [ ]:
# Install the *.deb file with guppy basecaller
sudo apt install ./ont_guppy_cpu_6.5.7-1~focal_amd64.deb

In [ ]:
# Verify that the environment is installed and working by showing the "version"
guppy_basecaller --version

In [ ]:
# Check the help menu for details
guppy_basecaller -h

In [ ]:
# Execute the guppy module

guppy_basecaller \
  -i ~/path/to/folder/input/pod5_files \
  -s ~/path/to/folder/guppy_output \
  -c dna_r9.4.1_450bps_hac.cfg \
  --num_callers 4 \
  --cpu_threads_per_caller 2

Explanation of the script:

1. guppy_basecaller = calls the guppy module
2. -i = path to input folder
3. -s = path to save folder with fastq data created
4. -c = flow cell and chemistry configuration (R9.4.1, an older chemistry; current ONT data uses R10.4.1 and the Dorado basecaller, which replaced Guppy). The "hac" indicates "high accuracy" (good quality, slow process)
5. --num_callers = number of parallel processes
6. --cpu_threads_per_caller number of cores per caller

The process using only CPU will take some days to finish, depending on the file size.

In [ ]:
# Deactivate the guppy environment
conda deactivate

## Compress the multi-fastq files created into a gz file using gzip for downstream analysis

This is to use the porechop tool

In [ ]:
# compress files using gzip
gzip ~/path/to/file/*.fastq

## Quality analysis

This section is to verify the quality of the fastq files generated, data analysis of the summary_sequencing.txt file generated by guppy.

In [ ]:
# Install nanoplot tool
pip install NanoPlot

In [ ]:
# Create plots from sequencing using the summary file from guppy basecaller
NanoPlot \
--summary sequencing_summary.txt \
--loglength \
-o ~/path/to/folder/output 

## Trimming and QC filter of fastq files

### Trimming with porechop
The trimming of the ONT adapters will be made with the module porechop

In [ ]:
# Create the environment
conda create -n ont-trimming -y

In [ ]:
# Activate environment
conda activate ont-trimming

In [ ]:
# install porechop
mamba install porechop

Trimming the sequences to eliminate the adapters added during the ONT sequencing.

In [ ]:
# Install Filtlong to select the high-quality readings after trimming
conda install -c bioconda filtlong

In [ ]:
# Run porechop in a for loop to keep multi-fastq trimmed files
for file in *.fastq.gz; do porechop -i "$file" -o ~/path/to/output/file/"trimmed_${file}"; done

### QC filtering
For this section, look into the plots generated by NanoPlot and select the parameters to filter high-quality sequences using Filtlong.
Based on the information, filter the data based on the following:
1. Minimum Quality = 10
2. Minimum length (bp) = 900
3. Discard the worst 10%

In [ ]:
# Install filtlong tool
conda install -c bioconda filtlong -y

#### Filtering with filtlong script for linux
Memory Management
1. 4 parallel jobs maximum: With Filtlong using approximately 1-6GB RAM per process, running 4 jobs simultaneously uses ~24GB, leaving 8GB for the system
2. Memory limits per process: Uses ulimit to cap memory usage per job at 6GB
3. Temporary file management: Uses /tmp for intermediate files and cleans up automatically

CPU Utilization
1. 50% core utilization: Uses 4 out of 8 cores, preventing system overload
2. Staggered job starts: 2-second delay between job launches to prevent resource spikes
3. Active monitoring: Tracks memory usage and job completion in real-time
   
Performance Features
1. Batch processing: Maintains consistent parallel load without overwhelming the system
2. Resource monitoring: Real-time memory usage tracking
3. Error handling: Comprehensive logging and failure detection
4. Automatic cleanup: Removes temporary files and handles interrupted processing

#### Adjust Filtlong parameters:
The script (scripts/enhanced_filtlong_combined_nanoplot.sh) uses these parameters; set the input and output directories at the top:\
KEEP_PERCENT = 90: Keep 90% of bases (by quality)\
TARGET_BASES = 500000000: Target 500M bases output\
MIN_LENGTH = 900: Minimum read length 0.9 kb\
MIN_MEAN_Q = 10: Minimum mean quality score (Q10)

In [ ]:
# To run the script in bash
chmod +x enhanced_filtlong_combined_nanoplot.sh
./enhanced_filtlong_combined_nanoplot.sh

#### Enhanced script with QC filtering combined with Post-filtering QC analysis with NanoPlot

This script adds a post-filtering QC analysis to the files generated by filtlong using nanoplot.
The output structure of the file directory will be as follows:

In [ ]:
output_directory/
├── filtered/                           # Individual filtered files
│   ├── sample1_filtered.fastq.gz
│   ├── sample2_filtered.fastq.gz
│   └── sample3_filtered.fastq.gz
├── combined_qc/                        # Single combined QC analysis
│   ├── Combined_Dataset_NanoPlot-report.html  # Main interactive report
│   ├── NanoStats.txt                          # Combined statistics
│   ├── Combined_Dataset_combined_summary.txt  # Enhanced summary
│   └── individual_plots/                      # PNG and HTML plots
├── logs/                               # Processing logs
│   ├── sample1_filtlong.log
│   ├── sample2_filtlong.log
│   └── combined_nanoplot.log           # Combined QC log
└── pipeline_summary.txt                # Overall pipeline summary


Script for bash:

In [ ]:
# The full script is in scripts/enhanced_filtlong_combined_nanoplot.sh
# Edit INPUT_DIR and OUTPUT_DIR at the top of the script, then run:
bash scripts/enhanced_filtlong_combined_nanoplot.sh

In [ ]:
# Deactivate environment
conda deactivate

In [ ]:
# Change the long names of each file to a short "trimmed_QC-filter_{#}".fastq.gz
#locate the folder with the files
cd ~/path/to/folder
counter=0
folder_name=$(basename "$PWD")
for file in *; do
    if [ -f "$file" ]; then
        # Get file extension
        ext="${file##*.}"
        # Only add extension if file had one
        if [ "$ext" != "$file" ]; then
            mv "$file" "${folder_name}_${counter}.fastq.${ext}"
        else
            mv "$file" "${folder_name}_${counter}"
        fi
        ((counter++))
    fi
done

## Metagenomic and taxonomic analysis using Kraken2/Bracken and Krona

### Create the metagenomics environment with conda

In [ ]:
# Create the environment metagenomics
conda create -n metagenomics -y

# Activate the environment
conda activate metagenomics

# Add biopython to avoid errors to install the dependencies
conda install -c conda-forge biopython parallel -y

# Add the metagenomics dependencies
conda install -c bioconda kraken2 bracken krakentools krona pandas -y

### Download a pre-built standard database for kraken2 analysis

For this analysis we will use the PlusPFP-16 data set from kraken2 data sets:\
https://benlangmead.github.io/aws-indexes/k2 

The PlusPFP-16 is the best choice for broad eDNA metagenomics including plants, fungi, protists, bacteria, and viruses. It is designed to use with a computer with limited RAM memory (16 GB). This will be done in a computer with 32 GB of RAM.

Steps:
1. Download the PlusPFP data set .tar.gz to the computer
2. Extract the file

### Kraken2 shotgun metagenomic classification

`scripts/kraken2_batch_analysis.sh` classifies the 25 filtered, high-quality FASTQ.gz files with Kraken2.

**Design choices**
- **One combined run.** The files are concatenated and classified in a single Kraken2 run, so the ~16 GB database is loaded into RAM only once instead of 25 times. The files are treated as technical replicates of the same environmental sample. The `--individual` flag also classifies each file separately for comparison.
- **Confidence threshold.** Set to 0.0, the Kraken2 default. For eDNA, raising it to 0.05–0.1 trades a little sensitivity for fewer false-positive assignments, so it is worth comparing both.
- **Minimum base quality 10**, consistent with the Filtlong Q10 filter upstream.
- **8 threads**, sized for an 8-core, 32 GB RAM workstation.

**Outputs**
- `combined_kraken2_report.txt`: per-taxon read counts, the input for Bracken and Krona
- `combined_kraken2_output.txt`: per-read classifications
- `classification_summary.txt`: total reads, classified and unclassified reads, and classification rate

**Downstream steps (next stage of the project)**

Bracken re-estimates abundance by redistributing reads assigned to higher taxonomic levels down to species or genus level:
```bash
bracken -d $KRAKEN2_DB -i combined_kraken2_report.txt -o combined_bracken_species.txt -r 1000 -l S
bracken -d $KRAKEN2_DB -i combined_kraken2_report.txt -o combined_bracken_genus.txt  -r 1000 -l G
```
(`-r` must match a read length the database was built for. ONT reads are long, so use the largest available.)

Krona turns the report into an interactive taxonomy chart:
```bash
python kreport2krona.py -r combined_kraken2_report.txt -o combined.krona
ktImportText combined.krona -o combined_taxonomy.html
```

#### Script usage
Edit the configuration block at the top of `scripts/kraken2_batch_analysis.sh`:

```bash
KRAKEN2_DB="/path/to/kraken2_db/k2_pluspfp_16gb"
INPUT_DIR="/path/to/filtered/fastq/files"
OUTPUT_DIR="/path/to/kraken2/output"
THREADS=8
```

In [ ]:
# Make the script executable
chmod +x scripts/kraken2_batch_analysis.sh

In [ ]:
# Run the combined analysis
./scripts/kraken2_batch_analysis.sh

### Running the script

In [ ]:
# The full script is in scripts/kraken2_batch_analysis.sh
# Edit KRAKEN2_DB, INPUT_DIR and OUTPUT_DIR at the top of the script, then run:
bash scripts/kraken2_batch_analysis.sh                 # combined analysis
bash scripts/kraken2_batch_analysis.sh --individual    # also classify each file separately